# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print metadata information
print(f"{metadata.name}: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Keywords: {', '.join(metadata.keywords)}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get available record sets
record_sets = dataset.metadata.recordSet
# If there are no record sets in metadata (but some in schema), try fallback
if not record_sets:
    # Fallback: query the dataset object
    record_sets = list(dataset.record_sets.keys())
else:
    record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in record_sets]
print("Available Record Sets (@id):")
for rs_id in record_sets:
    print(f"  - {rs_id}")

# For each record set, list fields
for rs_id in record_sets:
    record_set_obj = dataset.record_sets[rs_id]
    print(f"\nFields in Record Set {rs_id}:")
    for field in record_set_obj.fields:
        print(f"  - Field Name: {field.name} (@id: {field['@id'] if '@id' in field else getattr(field, '@id', 'N/A')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract records from all record sets
dataframes = {}

for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nLoaded {len(df)} records from RecordSet {rs_id}.")
        if not df.empty:
            print(f"Columns (@id): {list(df.columns)}")
            print(df.head())
    except Exception as e:
        print(f"Failed loading {rs_id}: {e}")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose the main survey RecordSet by inspecting overview above
# Example: If the main RecordSet is 'https://api.app.sen.science/frontiers/7160411/e1f3c048-91dc-444a-b31e-e649e1eb302f/records' (replace with actual @id!)
main_record_set_id = record_sets[0] if record_sets else None
df = dataframes.get(main_record_set_id)

# Inspect columns to find numeric fields (e.g., GAD-7, PHQ-9 total scores)
numeric_fields = [col for col in df.columns if 'GAD' in col or 'PHQ' in col or 'PSQ' in col or 'score' in col]
print(f"Numeric fields detected: {numeric_fields}")

# Let's pick a numeric field, e.g., 'GAD7_score' or similar
numeric_field = numeric_fields[0] if numeric_fields else None
if numeric_field:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a key demographic, e.g. 'level_of_education' or 'gender'
    group_fields = [col for col in df.columns if 'education' in col or 'gender' in col or 'village' in col or 'age_group' in col]
    group_field = group_fields[0] if group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric field detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Visualize numeric field distribution
if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(f'{numeric_field}')
    plt.ylabel('Frequency')
    plt.show()
    
    # If group_field available, plot mean scores per group
    if group_field:
        plt.figure(figsize=(8, 4))
        group_means = df.groupby(group_field)[numeric_field].mean().dropna()
        group_means.plot(kind='bar')
        plt.title(f'Mean {numeric_field} per {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.
- Loaded metadata and survey records using `mlcroissant`.
- Reviewed demographic and mental health indicators (e.g., GAD-7, PHQ-9 scores).
- Filtered and normalized fields for exploratory analysis.
- Grouped data by relevant attributes (e.g., education or gender).
- Visualized distributions to aid insight into potential mental health patterns.

Further analysis can explore correlations, missing data handling, or cross-sectional comparisons between demographic groups.